# 📧 Email Spam Classification — NLP Naive Bayes

**Dataset:** SpamAssassin (5,796 emails — 3,900 ham / 1,896 spam)  
**Model:** Naive Bayes + TF-IDF (1,2)-grams + Hand-Crafted Features

### Key Design Decisions
| Concern | Solution |
|---|---|
| **Data Leakage** | Train/test split BEFORE feature fitting; `Pipeline` enforces fit-on-train-only |
| **Overfitting** | StratifiedKFold CV, L2 regularisation | 
| **Class Imbalance** | `class_weight='balanced'` + stratified splits |

### Table of Contents
1. [Imports & Setup](#1)
2. [Load & Explore Data](#2)
3. [Text Preprocessing](#3)
4. [Hand-Crafted Feature Engineering](#4)
5. [Train / Test Split](#5)
6. [Combined Pipeline](#6)
7. [Cross-Validation](#7)
8. [Hyperparameter Tuning](#8)
9. [Learning Curves](#9)
10. [Final Evaluation](#10)
11. [Feature Importance](#11)
12. [Summary](#12)


---
## 1. Imports & Setup <a id='1'></a>

In [25]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import hstack, csr_matrix

import mlflow
import mlflow.sklearn

from sklearn.compose                 import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline                import Pipeline
from sklearn.base                    import BaseEstimator, TransformerMixin
from sklearn.preprocessing           import MinMaxScaler
from sklearn.naive_bayes             import MultinomialNB
from sklearn.model_selection         import (
    train_test_split, StratifiedKFold,
    cross_validate, learning_curve, GridSearchCV,
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, ConfusionMatrixDisplay, roc_curve, auc,
    accuracy_score, f1_score, precision_score, recall_score,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
SEED = 42
print('Imports ready ✅')

Imports ready ✅


---
## 2. Load & Explore Data <a id='2'></a>

In [26]:
df = pd.read_csv('../data/spam_assassin.csv')

print('Shape   :', df.shape)
print('Columns :', df.columns.tolist())
print('Nulls:\n', df.isnull().sum())
df.head(3)

Shape   : (5796, 2)
Columns : ['text', 'target']
Nulls:
 text      0
target    0
dtype: int64


,text,target
0,From ilug-admin@linux.ie Mon Jul 29 11:28:02 2...,0
1,From gort44@excite.com Mon Jun 24 17:54:21 200...,1
2,From fork-admin@xent.com Mon Jul 29 11:39:57 2...,1


In [27]:
label_map = {0: 'Ham', 1: 'Spam'}
df['label'] = df['target'].map(label_map)
counts = df['label'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Class Distribution', fontweight='bold')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontweight='bold')

axes[1].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=['steelblue', 'tomato'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Proportion', fontweight='bold')

plt.tight_layout()
plt.show()

df['char_count'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()
print(df.groupby('label')[['char_count', 'word_count']].describe().round(1))
df.drop(columns=['char_count', 'word_count', 'label'], inplace=True)

      char_count                                                           \
           count    mean     std    min     25%     50%     75%       max   
label                                                                       
Ham       3900.0  3483.1  3134.1  362.0  2435.2  3196.5  4049.2   92469.0   
Spam      1896.0  5667.1  9068.5  736.0  2391.8  3824.0  6304.0  232305.0   

      word_count                                                    
           count   mean    std   min    25%    50%    75%      max  
label                                                               
Ham       3900.0  405.7  484.9  45.0  262.0  347.0  454.0  15164.0  
Spam      1896.0  545.6  612.2  73.0  250.0  386.0  617.2  11857.0  


In [28]:
def preprocess_email(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'^(from|subject|to|cc|bcc|received|content-type|mime-version|'
                  r'message-id|return-path|delivered-to|x-[a-z-]+):.*$',
                  '', text, flags=re.MULTILINE | re.IGNORECASE)
    text = re.sub(r'https?://\S+|www\.\S+', ' url ', text)
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.-]+', ' email ', text)
    text = re.sub(r'\$\s*\d+[\d,.]*', ' money ', text)
    text = re.sub(r'\b(\+?\d[\s.-]?){7,15}\b', ' phone ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(preprocess_email)

print('Original:')
print(df['text'].iloc[1][:250])
print('\nCleaned:')
print(df['clean_text'].iloc[1][:250])

Original:
From gort44@excite.com Mon Jun 24 17:54:21 2002 Return-Path: gort44@excite.com Delivery-Date: Tue Jun 4 05:31:16 2002 Received: from mandark.labs.netnoteinc.com ([213.105.180.140]) by dogma.slashnull.org (8.11.6/8.11.6) with ESMTP id g544VFO20182 for

Cleaned:
from email mon jun return path email delivery date tue jun received from mandark labs netnoteinc com phone by dogma slashnull org with esmtp id g vfo for tue jun received from wi poli poli cl phone by mandark labs netnoteinc com with smtp id g vc tue


---
## 3. Text Preprocessing <a id='3'></a>

> Purely rule-based — no statistics learned. Safe to apply before the split.


In [29]:
def extract_features(series: pd.Series) -> pd.DataFrame:
    raw   = series.fillna('')
    words = raw.str.split()
    return pd.DataFrame({
        'caps_ratio'       : raw.apply(lambda t: sum(c.isupper() for c in t) / max(len(t), 1)),
        'exclamation_count': raw.str.count(r'!'),
        'url_count'        : raw.str.count(r'https?://|www\.'),
        'dollar_count'     : raw.str.count(r'\$'),
        'html_flag'        : raw.str.contains(r'<html|<body|<table|<td|<font',
                                              case=False, regex=True).astype(int),
        'word_count'       : words.str.len().fillna(0),
        'avg_word_length'  : words.apply(
                                lambda ws: np.mean([len(w) for w in ws])
                                if isinstance(ws, list) and ws else 0),
        'digit_ratio'      : raw.apply(lambda t: sum(c.isdigit() for c in t) / max(len(t), 1)),
        'unique_word_ratio': words.apply(
                                lambda ws: len(set(ws)) / max(len(ws), 1)
                                if isinstance(ws, list) and ws else 0),
    }, index=series.index).astype(float)


feat_df        = extract_features(df['text'])
df             = pd.concat([df, feat_df], axis=1)
HAND_FEAT_COLS = feat_df.columns.tolist()

print('Hand-crafted features:', HAND_FEAT_COLS)
print()
print(feat_df.describe().round(3).to_string())

Hand-crafted features: ['caps_ratio', 'exclamation_count', 'url_count', 'dollar_count', 'html_flag', 'word_count', 'avg_word_length', 'digit_ratio', 'unique_word_ratio']

       caps_ratio  exclamation_count  url_count  dollar_count  html_flag  word_count  avg_word_length  digit_ratio  unique_word_ratio
count    5796.000           5796.000   5796.000      5796.000   5796.000    5796.000         5796.000     5796.000           5796.000
mean        0.075              2.578      6.277         1.751      0.180     451.448            8.513        0.104              0.636
std         0.051              6.854     19.346         8.594      0.384     533.936            3.429        0.032              0.078
min         0.019              0.000      0.000         0.000      0.000      45.000            4.095        0.005              0.150
25%         0.054              0.000      2.000         0.000      0.000     258.750            7.121        0.085              0.598
50%         0.063        

In [30]:
label_map   = {0: 'Ham', 1: 'Spam'}
df['_label'] = df['target'].map(label_map)

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()

for ax, col in zip(axes, HAND_FEAT_COLS):
    for lbl, color in [('Ham', 'steelblue'), ('Spam', 'tomato')]:
        subset = df[df['_label'] == lbl][col]
        subset.clip(upper=subset.quantile(0.98)).plot(
            kind='hist', bins=40, ax=ax, alpha=0.6,
            color=color, label=lbl, edgecolor='white', density=True)
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Hand-Crafted Feature Distributions: Ham vs Spam',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

df.drop(columns=['_label'], inplace=True)

In [31]:
X_text  = df['clean_text']
X_feats = df[HAND_FEAT_COLS]
y       = df['target']

(X_train_text, X_test_text,
 X_train_feats, X_test_feats,
 y_train, y_test) = train_test_split(
    X_text, X_feats, y,
    test_size    = 0.20,
    random_state = SEED,
    stratify     = y,
)

print(f'Train : {X_train_text.shape[0]} samples')
print(f'Test  : {X_test_text.shape[0]} samples')
print('\nTrain class split:')
print(y_train.value_counts(normalize=True).rename({0:'Ham',1:'Spam'}).map('{:.1%}'.format))
print('\nTest class split:')
print(y_test.value_counts(normalize=True).rename({0:'Ham',1:'Spam'}).map('{:.1%}'.format))
print('\n✅ Test set locked — not touched until Section 10')

Train : 4636 samples
Test  : 1160 samples

Train class split:
target
Ham     67.3%
Spam    32.7%
Name: proportion, dtype: object

Test class split:
target
Ham     67.3%
Spam    32.7%
Name: proportion, dtype: object

✅ Test set locked — not touched until Section 10


In [32]:
# ─────────────────────────────────────────────────────────────
# Section 6 — Combined Pipeline
# TF-IDF (1,2)-grams on clean_text + MinMaxScaled hand-crafted
# features → MultinomialNB.  ColumnTransformer ensures the
# TF-IDF is *only* fit on the training fold in each CV split
# (no data leakage).
# ─────────────────────────────────────────────────────────────

TFIDF_PARAMS = dict(
    ngram_range  = (1, 2),
    max_features = 50_000,
    min_df       = 2,
    sublinear_tf = True,
)

def make_pipeline(alpha=0.1, fit_prior=True):
    """
    ColumnTransformer routes:
      • 'clean_text' column  → TfidfVectorizer  (sparse)
      • HAND_FEAT_COLS       → MinMaxScaler     (dense → sparse via remainder)
    Both are stacked horizontally then fed to MultinomialNB.
    MinMaxScaler guarantees values ≥ 0, which MultinomialNB requires.
    """
    ct = ColumnTransformer(
        transformers=[
            ('tfidf', TfidfVectorizer(**TFIDF_PARAMS), 'clean_text'),
            ('hand',  MinMaxScaler(),                   HAND_FEAT_COLS),
        ],
        sparse_threshold=0.0,   # keep dense output from MinMaxScaler
    )
    return Pipeline([
        ('features', ct),
        ('clf',      MultinomialNB(alpha=alpha, fit_prior=fit_prior)),
    ])

# Combine text + hand-crafted features into a single DataFrame
# (ColumnTransformer selects columns by name, so DataFrame is needed)
X_train_combined = pd.concat([X_train_text, X_train_feats], axis=1)
X_test_combined  = pd.concat([X_test_text,  X_test_feats],  axis=1)

print(f'X_train_combined shape : {X_train_combined.shape}')
print(f'X_test_combined  shape : {X_test_combined.shape}')
print(f'Columns : clean_text + {HAND_FEAT_COLS}')
print('✅ Pipeline factory and combined feature matrices ready')

X_train_combined shape : (4636, 10)
X_test_combined  shape : (1160, 10)
Columns : clean_text + ['caps_ratio', 'exclamation_count', 'url_count', 'dollar_count', 'html_flag', 'word_count', 'avg_word_length', 'digit_ratio', 'unique_word_ratio']
✅ Pipeline factory and combined feature matrices ready


In [33]:


import dagshub

mlflow.set_tracking_uri('https://dagshub.com/kaushik-chariya/Deep-Shield-Mail.mlflow')
dagshub.init(repo_owner='kaushik-chariya', repo_name='Deep-Shield-Mail', mlflow=True)


mlflow.set_experiment("Naive-Bayes-Spam-Detection-HyperParamater")

Initialized MLflow to track repo "kaushik-chariya/Deep-Shield-Mail"

Repository kaushik-chariya/Deep-Shield-Mail initialized!

<Experiment: artifact_location='mlflow-artifacts:/eda601eab04e43e9b6df9444af0451ac', creation_time=1779060462320, experiment_id='14', last_update_time=1779060462320, lifecycle_stage='active', name='Naive-Bayes-Spam-Detection-HyperParamater', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [34]:
# ─────────────────────────────────────────────────────────────
# Section 7 — Cross-Validation  

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

with mlflow.start_run(run_name="NaiveBayes-CombinedFeatures-CV") as cv_run:

    mlflow.set_tags({
        'model'   : 'MultinomialNB',
        'features': 'tfidf + hand-crafted',
        'stage'   : 'cross-validation',
        'dataset' : 'SpamAssassin',
        'note'    : 'MinMaxScaler keeps values ≥ 0 for MultinomialNB',
    })

    mlflow.log_params({
        'alpha'             : 0.1,
        'fit_prior'         : True,
        'scaler'            : 'MinMaxScaler',
        'tfidf_ngram_range' : str(TFIDF_PARAMS['ngram_range']),
        'tfidf_max_features': TFIDF_PARAMS['max_features'],
        'tfidf_min_df'      : TFIDF_PARAMS['min_df'],
        'tfidf_sublinear_tf': TFIDF_PARAMS['sublinear_tf'],
        'hand_feat_count'   : len(HAND_FEAT_COLS),
        'cv_folds'          : 5,
        'train_size'        : len(y_train),
    })

    scores = cross_validate(
        make_pipeline(),
        X_train_combined, y_train,
        cv                = skf,
        scoring           = ['accuracy', 'f1', 'precision', 'recall', 'roc_auc'],
        return_train_score= True,
        n_jobs            = -1,
    )

    train_f1 = scores['train_f1'].mean()
    val_f1   = scores['test_f1'].mean()
    gap      = train_f1 - val_f1

    mlflow.log_metrics({
        'cv_accuracy'    : scores['test_accuracy'].mean(),
        'cv_accuracy_std': scores['test_accuracy'].std(),
        'cv_f1'          : val_f1,
        'cv_f1_std'      : scores['test_f1'].std(),
        'cv_precision'   : scores['test_precision'].mean(),
        'cv_recall'      : scores['test_recall'].mean(),
        'cv_auc_roc'     : scores['test_roc_auc'].mean(),
        'cv_auc_roc_std' : scores['test_roc_auc'].std(),
        'cv_train_f1'    : train_f1,
        'cv_overfit_gap' : gap,
    })

    print('Naive Bayes — 5-Fold CV Results')
    print('=' * 45)
    print(f'  Accuracy  : {scores["test_accuracy"].mean():.4f} ± {scores["test_accuracy"].std():.4f}')
    print(f'  F1        : {val_f1:.4f} ± {scores["test_f1"].std():.4f}')
    print(f'  Precision : {scores["test_precision"].mean():.4f}')
    print(f'  Recall    : {scores["test_recall"].mean():.4f}')
    print(f'  AUC-ROC   : {scores["test_roc_auc"].mean():.4f}')
    print(f'  Train F1  : {train_f1:.4f}')
    print(f'  Overfit   : {gap:.4f}  {"⚠️ HIGH" if gap > 0.05 else "✅ OK"}')
    print(f'\n  MLflow Run ID : {cv_run.info.run_id}')
    print('✅ CV run logged to MLflow')

Naive Bayes — 5-Fold CV Results
  Accuracy  : 0.9778 ± 0.0036
  F1        : 0.9651 ± 0.0057
  Precision : 0.9916
  Recall    : 0.9400
  AUC-ROC   : 0.9989
  Train F1  : 0.9688
  Overfit   : 0.0036  ✅ OK

  MLflow Run ID : 701742099bf44287900bbad0e68a4c31
✅ CV run logged to MLflow


In [35]:
# ─────────────────────────────────────────────────────────────
# Section 8 — Hyperparameter Tuning (GridSearchCV)  (MLflow Run)
# ─────────────────────────────────────────────────────────────


with mlflow.start_run(run_name="NaiveBayes-GridSearchCV-Alpha") as gs_run:

    mlflow.set_tags({
        'model'   : 'MultinomialNB',
        'features': 'tfidf + hand-crafted',
        'stage'   : 'hyperparameter-tuning',
        'dataset' : 'SpamAssassin',
    })

    param_grid = {'clf__alpha': [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0]}

    mlflow.log_params({
        'param_grid': str(param_grid['clf__alpha']),
        'scoring'   : 'precision',
        'cv_folds'  : 5,
        'refit'     : True,
        'train_size': len(y_train),
    })

    grid_search = GridSearchCV(
        make_pipeline(),
        param_grid,
        cv      = skf,
        scoring = 'precision',
        n_jobs  = -1,
        verbose = 1,
        refit   = True,
    )
    grid_search.fit(X_train_combined, y_train)

    best_alpha = grid_search.best_params_['clf__alpha']
    best_score = grid_search.best_score_
    best_model = grid_search.best_estimator_

    mlflow.log_params({'best_alpha': best_alpha})
    mlflow.log_metrics({'best_cv_precision': best_score})

    # ── Save and log the alpha-tuning plot ────────────────────────
    results_gs = pd.DataFrame(grid_search.cv_results_)
    fig_gs, ax_gs = plt.subplots(figsize=(8, 4))
    ax_gs.plot(results_gs['param_clf__alpha'], results_gs['mean_test_score'],
               'o-', color='steelblue', linewidth=2, markersize=7)
    ax_gs.fill_between(
        results_gs['param_clf__alpha'],
        results_gs['mean_test_score'] - results_gs['std_test_score'],
        results_gs['mean_test_score'] + results_gs['std_test_score'],
        alpha=0.15, color='steelblue',
    )
    ax_gs.axvline(best_alpha, color='tomato', linestyle='--',
                  label=f'Best alpha = {best_alpha}')
    ax_gs.set_xscale('log')
    ax_gs.set_xlabel('Alpha (Laplace smoothing)')
    ax_gs.set_ylabel('CV Precision')
    ax_gs.set_title('GridSearchCV — Alpha Tuning', fontweight='bold')
    ax_gs.legend()
    fig_gs.tight_layout()
    mlflow.log_figure(fig_gs, 'plots/alpha_tuning.png')   # ← logged as artifact
    plt.show()

    mlflow.sklearn.log_model(
        best_model,
        artifact_path         = 'naive_bayes_best',
        registered_model_name = 'spam_naive_bayes_combined',
    )

    print(f'\n✅ Best alpha       : {best_alpha}')
    print(f'   Best CV Precision : {best_score:.4f}')
    print(f'   MLflow Run ID     : {gs_run.info.run_id}')
    print('✅ GridSearch run logged to MLflow')

Fitting 5 folds for each of 7 candidates, totalling 35 fits


Successfully registered model 'spam_naive_bayes_combined'.
2026/05/18 05:47:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: spam_naive_bayes_combined, version 1
Created version '1' of model 'spam_naive_bayes_combined'.



✅ Best alpha       : 0.001
   Best CV Precision : 0.9979
   MLflow Run ID     : 86c77926ed254fdc9ac083d808b33bf2
✅ GridSearch run logged to MLflow


In [36]:
# ─────────────────────────────────────────────────────────────
# Sections 9–11 — Learning Curves, Final Test Evaluation &
#                 Feature Importance        (MLflow Run)
# ─────────────────────────────────────────────────────────────
import os, tempfile

with mlflow.start_run(run_name="NaiveBayes-FinalEvaluation") as eval_run:

    mlflow.set_tags({
        'model'   : 'MultinomialNB',
        'features': 'tfidf + hand-crafted',
        'stage'   : 'final-evaluation',
        'dataset' : 'SpamAssassin',
    })
    mlflow.log_params({
        'best_alpha': grid_search.best_params_['clf__alpha'],
        'test_size' : len(y_test),
        'train_size': len(y_train),
    })

    # ══════════════════════════════════════════════════════════
    # Section 9 — Learning Curves
    # ══════════════════════════════════════════════════════════
    train_sizes, train_scores, val_scores = learning_curve(
        best_model,
        X_train_combined, y_train,
        cv          = skf,
        scoring     = 'f1',
        train_sizes = np.linspace(0.10, 1.0, 10),
        n_jobs      = -1,
    )
    train_mean, train_std = train_scores.mean(axis=1), train_scores.std(axis=1)
    val_mean,   val_std   = val_scores.mean(axis=1),   val_scores.std(axis=1)
    lc_gap = train_mean[-1] - val_mean[-1]

    fig_lc, ax_lc = plt.subplots(figsize=(9, 5))
    ax_lc.plot(train_sizes, train_mean, 'o-', color='steelblue', label='Train F1')
    ax_lc.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                       alpha=0.15, color='steelblue')
    ax_lc.plot(train_sizes, val_mean, 's-', color='tomato', label='Validation F1')
    ax_lc.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                       alpha=0.15, color='tomato')
    ax_lc.set_title(f'Learning Curve  |  Overfit gap: {lc_gap:.4f} '
                    f'{"⚠️ HIGH" if lc_gap > 0.05 else "✅ OK"}', fontweight='bold')
    ax_lc.set_xlabel('Training Samples')
    ax_lc.set_ylabel('F1 Score')
    ax_lc.legend()
    ax_lc.grid(True, alpha=0.3)
    fig_lc.tight_layout()
    mlflow.log_figure(fig_lc, 'plots/learning_curve.png')   # ← logged as artifact
    plt.show()

    mlflow.log_metric('lc_overfit_gap', lc_gap)

    # ══════════════════════════════════════════════════════════
    # Section 10 — Final Test Set Evaluation
    # ══════════════════════════════════════════════════════════
    y_pred  = best_model.predict(X_test_combined)
    y_proba = best_model.predict_proba(X_test_combined)[:, 1]

    test_acc  = accuracy_score(y_test, y_pred)
    test_f1   = f1_score(y_test, y_pred)
    test_prec = precision_score(y_test, y_pred)
    test_rec  = recall_score(y_test, y_pred)
    test_auc  = roc_auc_score(y_test, y_proba)

    mlflow.log_metrics({
        'test_accuracy' : test_acc,
        'test_f1'       : test_f1,
        'test_precision': test_prec,
        'test_recall'   : test_rec,
        'test_auc_roc'  : test_auc,
    })

    print('=' * 55)
    print('FINAL TEST SET RESULTS — Naive Bayes')
    print('=' * 55)
    print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))
    print(f'AUC-ROC : {test_auc:.4f}')

    # ── Confusion Matrix ───────────────────────────────────────
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    mlflow.log_metrics({'test_tp': tp, 'test_tn': tn,
                        'test_fp': fp, 'test_fn': fn})

    fig_cm, axes_cm = plt.subplots(1, 2, figsize=(13, 5))
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred),
        display_labels=['Ham', 'Spam']
    ).plot(ax=axes_cm[0], colorbar=False, cmap='Blues')
    axes_cm[0].set_title('Confusion Matrix (Counts)', fontweight='bold')
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred, normalize='true'),
        display_labels=['Ham', 'Spam']
    ).plot(ax=axes_cm[1], colorbar=False, cmap='Greens', values_format='.2%')
    axes_cm[1].set_title('Confusion Matrix (Normalised)', fontweight='bold')
    fig_cm.tight_layout()
    mlflow.log_figure(fig_cm, 'plots/confusion_matrix.png')  # ← logged as artifact
    plt.show()

    print(f'True Negatives  (Ham → Ham)   : {tn}')
    print(f'True Positives  (Spam → Spam) : {tp}')
    print(f'False Positives (Ham → Spam)  : {fp}  ← False Alarm')
    print(f'False Negatives (Spam → Ham)  : {fn}  ← Missed Spam')

    # ── ROC Curve ─────────────────────────────────────────────
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc_val = auc(fpr, tpr)

    fig_roc, ax_roc = plt.subplots(figsize=(7, 5))
    ax_roc.plot(fpr, tpr, color='steelblue', lw=2,
                label=f'Naive Bayes (AUC = {roc_auc_val:.4f})')
    ax_roc.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random (AUC = 0.50)')
    ax_roc.fill_between(fpr, tpr, alpha=0.10, color='steelblue')
    ax_roc.set_xlabel('False Positive Rate')
    ax_roc.set_ylabel('True Positive Rate')
    ax_roc.set_title('ROC Curve — Test Set', fontweight='bold')
    ax_roc.legend(loc='lower right')
    ax_roc.grid(True, alpha=0.3)
    fig_roc.tight_layout()
    mlflow.log_figure(fig_roc, 'plots/roc_curve.png')         # ← logged as artifact
    plt.show()

    # ══════════════════════════════════════════════════════════
    # Section 11 — Feature Importance
    # ══════════════════════════════════════════════════════════
    nb_clf      = best_model.named_steps['clf']
    tfidf       = best_model.named_steps['features'].transformers_[0][1]
    tfidf_names = np.array(tfidf.get_feature_names_out())

    spam_diff    = nb_clf.feature_log_prob_[1, :len(tfidf_names)] - \
                   nb_clf.feature_log_prob_[0, :len(tfidf_names)]
    top_spam_idx = np.argsort(spam_diff)[-20:][::-1]
    top_ham_idx  = np.argsort(spam_diff)[:20]

    fig_fi, axes_fi = plt.subplots(1, 2, figsize=(15, 7))
    for ax, idx, color, title in [
        (axes_fi[0], top_spam_idx, 'tomato',    '🚨 Top 20 Spam Words'),
        (axes_fi[1], top_ham_idx,  'steelblue', '✅ Top 20 Ham Words'),
    ]:
        words  = tfidf_names[idx]
        scores = np.abs(spam_diff[idx])
        ax.barh(range(len(words)), scores[::-1], color=color,
                edgecolor='white', alpha=0.85)
        ax.set_yticks(range(len(words)))
        ax.set_yticklabels(words[::-1], fontsize=10)
        ax.set_xlabel('|log P(Spam) − log P(Ham)|')
        ax.set_title(title, fontweight='bold', fontsize=12)
    fig_fi.suptitle('Naive Bayes — Feature Importance',
                    fontsize=13, fontweight='bold', y=1.01)
    fig_fi.tight_layout()
    mlflow.log_figure(fig_fi, 'plots/feature_importance.png')  # ← logged as artifact
    plt.show()

    # ── Log best model under this run too ─────────────────────
    mlflow.sklearn.log_model(
        best_model,
        artifact_path         = 'naive_bayes_final',
        registered_model_name = 'spam_naive_bayes_combined',
    )

    print(f'\n  MLflow Run ID : {eval_run.info.run_id}')
    print('✅ Final evaluation logged to MLflow')
    print('   Artifacts: confusion_matrix.png | roc_curve.png | learning_curve.png | feature_importance.png')

FINAL TEST SET RESULTS — Naive Bayes
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.99       781
        Spam       1.00      0.94      0.97       379

    accuracy                           0.98      1160
   macro avg       0.98      0.97      0.98      1160
weighted avg       0.98      0.98      0.98      1160

AUC-ROC : 0.9996
True Negatives  (Ham → Ham)   : 780
True Positives  (Spam → Spam) : 357
False Positives (Ham → Spam)  : 1  ← False Alarm
False Negatives (Spam → Ham)  : 22  ← Missed Spam


Registered model 'spam_naive_bayes_combined' already exists. Creating a new version of this model...
2026/05/18 05:49:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: spam_naive_bayes_combined, version 2
Created version '2' of model 'spam_naive_bayes_combined'.



  MLflow Run ID : 1210af5188a747b7a9de98121b49bb02
✅ Final evaluation logged to MLflow
   Artifacts: confusion_matrix.png | roc_curve.png | learning_curve.png | feature_importance.png
